In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
!uv --version
!git clone https://github.com/DeepWok/mase.git
%cd mase
!uv sync
import glob, site, sys

venv_site = glob.glob("/content/mase/.venv/lib/python*/site-packages")[0]
site.addsitedir(venv_site)
sys.path.insert(0, venv_site)

print("Using:", venv_site)

In [ ]:

!uv run pytest test/ir/graph/test_create_masegraph.py
!uv run python -c "import chop; import transformers; print('Environment ready!')"

Task 1: Per-layer width/frac_width search with LinearInteger
Task 2: Multi-precision search across all supported quantized layers

In [ ]:
import sys, os, importlib

_lab_dir = os.path.join(os.getcwd(), "labs", "4")
if not os.path.isdir(_lab_dir):
    _lab_dir = os.getcwd()
if _lab_dir not in sys.path:
    sys.path.insert(0, _lab_dir)

for _mod_name in sorted(k for k in sys.modules if k == "src" or k.startswith("src.")):
    del sys.modules[_mod_name]

from src import (
    prep_env,
    make_objective,
    run_search,
    restrict_search_space,
    build_quant_config,
    plot_curves,
    best_so_far_curve,
    best_so_far_curves_by_precision,
    analyse_study,
)

In [ ]:
dataset, tokenizer, search_space, layer_map, base_model = prep_env()

In [ ]:
task1_space = restrict_search_space(search_space, ["Linear", "LinearInteger"])
obj_1 = make_objective(
    dataset, tokenizer, base_model, layer_map, task1_space,
    build_quant_config_fn=build_quant_config,
)
study_1 = run_search(obj_1, name="task1_integer", n_trials=30, delete_existing=True)

plot_curves(
    {"Integer-only": best_so_far_curve(study_1)},
    title="Task 1: Cumulative Best Accuracy",
)

In [ ]:
obj_2 = make_objective(
    dataset, tokenizer, base_model, layer_map, search_space,
    build_quant_config_fn=build_quant_config,
)
study_2 = run_search(obj_2, name="task2_all", n_trials=50, delete_existing=True)

plot_curves(
    best_so_far_curves_by_precision(study_2),
    title="Task 2: Cumulative Best Accuracy per Precision",
)

In [ ]:
analyse_study("task2_all", save_dir="plots/")
